In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 05. RAG y agente

Recuperación evaluada por separado, filtro de relevancia tipo CRAG, agente único frente a multiagente y juez doble.

In [2]:
R_ = json.load(open(os.path.join(MODELOS, 'metricas_rag.json'), encoding='utf-8')); tabla(R_['recuperacion'])

,recall@4,mrr,n
denso,0.85,0.825,20.0
denso + filtro de modulo,0.95,0.888,20.0
bm25,0.75,0.712,20.0
bm25 + filtro de modulo,0.80,0.800,20.0
hibrido,0.80,0.738,20.0
hibrido + filtro de modulo,0.90,0.867,20.0


In [3]:
R_['filtro_relevancia']

{'umbral': 0.7057,
 'exactitud_balanceada': 1.0,
 'auc': 1.0,
 'similitud_con_respuesta': {'media': 0.7719100000000001, 'min': 0.7057},
 'similitud_fuera_del_indice': {'media': 0.6609, 'max': 0.7054},
 'n_con_respuesta': 20,
 'n_fuera': 8}

In [4]:
A = json.load(open(os.path.join(MODELOS, 'metricas_agente_plantillas.json'), encoding='utf-8'))
pd.DataFrame({m: A[m]['resumen'] for m in ('agente_unico', 'multiagente')}).round(3)

,agente_unico,multiagente
n,45.000,45.000
ruteo_correcto,0.978,0.978
respuestas_sin_cifras_inventadas,1.000,1.000
sin_frases_prohibidas,1.000,1.000
cierra_con_tu_decides,1.000,1.000
pedidos_prohibidos_rechazados,1.000,1.000
latencia_media_s,1.292,1.271
latencia_p95_s,3.558,3.496
llamadas_llm_por_respuesta,0.000,0.000
por_plantilla,1.000,1.000


In [5]:
A['agente_unico']['fallos_de_ruteo'], A['multiagente']['fallos_de_ruteo']

([['Explícame cómo funciona el crédito', 'educacion', 'ofertas']],
 [['Explícame cómo funciona el crédito', 'educacion', 'ofertas']])

In [6]:
print(open(os.path.join(RAIZ, 'docs', 'diagramas', 'grafo_multiagente.md'), encoding='utf-8').read()[:1800])

# Grafo del agente (multiagente)

Generado con `app.get_graph().draw_mermaid()` de LangGraph.

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	cargar_contexto(cargar_contexto)
	router(router)
	redactar(redactar)
	juez(juez)
	plantilla(plantilla)
	finalizar(finalizar)
	supervisor(supervisor)
	agente_pronostico(agente_pronostico)
	agente_salud(agente_salud)
	agente_clima(agente_clima)
	agente_tasas(agente_tasas)
	agente_recomendador(agente_recomendador)
	agente_educador(agente_educador)
	__end__([<p>__end__</p>]):::last
	__start__ --> cargar_contexto;
	agente_clima --> supervisor;
	agente_educador --> supervisor;
	agente_pronostico --> supervisor;
	agente_recomendador --> supervisor;
	agente_salud --> supervisor;
	agente_tasas --> supervisor;
	cargar_contexto --> router;
	juez -.-> finalizar;
	juez -.-> plantilla;
	juez -.-> redactar;
	plantilla --> finalizar;
	redactar --> juez;
	router --> supervisor;
	supervisor -. &nbsp;clim

## Demo del agente

In [7]:
os.environ['LLM_PROVIDER'] = 'plantillas'
from src.agent.repositorio import Repositorio
from src.agent.grafo import construir_grafo, responder
from src.rag.indice import Buscador
sys.path.insert(0, os.path.join(RAIZ, 'eval'))
from evaluar_agente import clientes_de_prueba
repo = Repositorio(); ids = clientes_de_prueba(repo); app = construir_grafo(repo, Buscador(), 'agente_unico')
r = responder(app, ids['independiente_en_zona_del_nino'], '¿Estoy en riesgo por el Fenómeno El Niño?')
print(r['respuesta']); print(r['trazas'], r['cifras_verificadas'], r['rag_estado'])

Hola, Maestro.
En 30 dias esperamos que tu saldo ronde S/ 7,281, con un rango entre S/ 3,912 y S/ 14,733.
Para 2025-11 estimamos ingresos de S/ 4,447 (entre S/ 987 y S/ 9,383) y gastos de S/ 3,450.
Tu salud financiera es 72 de 100 y tu nivel es Maestro.
Lo que mas la baja es tu colchon (meses de ingreso disponibles).
Para subir a campeon te falta mejorar salud financiera: hoy tienes 72.1 y necesitas 75.
Alerta: tu zona (Piura) tiene exposicion alta a lluvias por El Nino segun el ENFEN. Acciones: Revisar el techo y las canaletas de la vivienda y reparar filtraciones antes de las lluvias. Tener un fondo de emergencia para al menos un mes de gastos. Guardar documentos importantes en un lugar alto y en copia digital.
La tasa de referencia del BCRP esta en 4.25% (sin cambio 0.00 puntos frente al mes anterior).
- Vives en Piura, zona con exposicion alta a lluvias por El Nino: prepara tu hogar y un fondo de S/ 7,144. Impacto estimado: S/ 7,144. Por que: el ENFEN mantiene la alerta de El Nino 